# LightRAG Hybrid Search Demo
## Vector + BM25 Keyword Search with RRF Fusion

파이프라인 각 단계를 셀별로 실행하여 중간 결과를 확인합니다.

1. **환경 설정** — 로컬 임베딩 + 원격 LLM
2. **LightRAG 선언**
3. **문서 로딩**
4. **Step 1: 청킹** — 원문을 청크로 분할, 결과 확인
5. **Step 2: 임베딩** — 청크를 벡터로 변환, VDB 저장
6. **Step 3: LLM 엔티티 추출** — 엔티티/릴레이션 추출
7. **Step 4: BM25 인덱싱** — 키워드 검색 인덱스 구축
8. **3가지 모드 쿼리 비교** — vector_only / keyword_only / hybrid

## 1. 환경 설정

In [ ]:
import sys, os, shutil, json
import numpy as np
sys.path.insert(0, '..')

from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc

LLM_BASE_URL = 'http://222.117.133.162:30010/v1'
LLM_MODEL    = 'qwen-task-pool'
LLM_API_KEY  = 'asdf'

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM = 384

# Load local embedding model (CPU, ~80MB)
print(f'Loading local embedding model: {EMBED_MODEL_NAME} ...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print('Embedding model loaded.')

async def llm_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return await openai_complete_if_cache(
        LLM_MODEL, prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        **kwargs,
    )

async def embed_func(texts):
    return embed_model.encode(texts, normalize_embeddings=True)

print(f'LLM: {LLM_BASE_URL} ({LLM_MODEL})')
print(f'Embedding: {EMBED_MODEL_NAME} (local, {EMBED_DIM}-dim)')

## 2. LightRAG 선언

In [ ]:
WORK_DIR = '/tmp/lightrag_hybrid_demo'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

rag = LightRAG(
    working_dir=WORK_DIR,
    llm_model_func=llm_func,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=8192,
        func=embed_func,
    ),
    addon_params={
        'enable_hybrid_search': True,
        'hybrid_search_mode': 'hybrid',
    },
)

await rag.initialize_storages()

print(f'working_dir: {WORK_DIR}')
print(f'enable_hybrid_search: {rag._addon_params["enable_hybrid_search"]}')
print(f'hybrid_search_mode:   {rag._addon_params["hybrid_search_mode"]}')

## 3. 문서 로딩

레포의 `docs/` 폴더에서 영문 마크다운 파일을 읽어옵니다.

In [ ]:
DOCS_DIR = os.path.join('..', 'docs')

md_files = []
for f in os.listdir(DOCS_DIR):
    if f.endswith('.md') and '-zh' not in f:
        path = os.path.join(DOCS_DIR, f)
        md_files.append((f, os.path.getsize(path)))

md_files.sort(key=lambda x: x[1], reverse=True)
# 작은 파일 1개로 빠르게 테스트 (큰 파일은 LLM 타임아웃 위험)
selected = md_files[-2:-1]  # FrontendBuildGuide.md (~5.6KB)

documents = []
file_paths = []
for fname, size in selected:
    path = os.path.join(DOCS_DIR, fname)
    with open(path, 'r') as f:
        text = f.read()
    documents.append(text)
    file_paths.append(fname)
    print(f'  {fname:45s} {size:>8,} bytes  ({len(text):,} chars)')

print(f'\nTotal: {len(documents)} documents, {sum(len(d) for d in documents):,} chars')

## Step 1: 청킹

LightRAG 내부 청킹 함수(`chunking_by_token_size`)를 직접 호출하여 결과를 확인합니다.
LLM이나 임베딩 호출 없이 즉시 실행됩니다.

In [ ]:
from lightrag.chunker import chunking_by_token_size
import tiktoken

tokenizer = tiktoken.encoding_for_model('gpt-4o')

all_chunks = []
for doc_text, fname in zip(documents, file_paths):
    chunks = chunking_by_token_size(
        tokenizer, doc_text,
        chunk_token_size=1200,
        chunk_overlap_token_size=100,
    )
    for c in chunks:
        c['file_path'] = fname
    all_chunks.extend(chunks)

print(f'=== 청킹 결과: {len(all_chunks)} chunks ===\n')
for i, c in enumerate(all_chunks):
    tokens = c['tokens']
    content = c['content']
    print(f'[Chunk {i+1}/{len(all_chunks)}] tokens={tokens}, file={c["file_path"]}')
    print(f'  {content[:120]}...')
    print()

## Step 2: 임베딩

청크 텍스트를 로컬 임베딩 모델로 벡터화합니다. (CPU, 수 초 소요)

In [ ]:
import time

texts = [c['content'] for c in all_chunks]

print(f'Embedding {len(texts)} chunks with {EMBED_MODEL_NAME} ...')
t0 = time.time()
vectors = embed_model.encode(texts, normalize_embeddings=True)
elapsed = time.time() - t0

print(f'Done in {elapsed:.1f}s')
print(f'Vector shape: {vectors.shape}  (chunks × dims)')
print()
for i in range(min(3, len(vectors))):
    print(f'  Chunk {i+1}: [{vectors[i][:5]}...]  (norm={np.linalg.norm(vectors[i]):.4f})')

## Step 2a: BM25 키워드 인덱싱 미리보기

청크 텍스트로 BM25 인덱스를 직접 빌드하여 키워드 검색 결과를 확인합니다.
LLM 호출 없이 즉시 실행됩니다.

In [ ]:
from lightrag.bm25_index import BM25Index

chunk_docs = {f'chunk-{i}': c['content'] for i, c in enumerate(all_chunks)}
preview_bm25 = BM25Index()
preview_bm25.build(chunk_docs)

print(f'BM25 index built: {len(preview_bm25.corpus_ids)} chunks\n')

test_queries = ['build frontend', 'bun install', 'WebUI', 'production']
print('=== BM25 키워드 검색 미리보기 ===')
for q in test_queries:
    hits = preview_bm25.query(q, top_k=3)
    print(f'\n  "{q}" → {len(hits)} hits')
    for r in hits:
        print(f'    [{r["id"]}] score={r["score"]:.2f}  {r["content"][:70]}...')

## Step 3: LLM 엔티티/릴레이션 추출 + VDB 저장 + BM25 인덱싱

`rag.ainsert()`를 실행하여 전체 파이프라인을 돌립니다.
위 Step 1~2에서 확인한 청킹/임베딩은 내부에서 자동으로 다시 수행됩니다.
BM25 인덱싱은 각 VDB upsert 시점마다 점진적으로 업데이트됩니다.

In [ ]:
track_id = await rag.ainsert(documents, file_paths=file_paths)
print(f'Insert complete (track_id: {track_id})')
print()

print('=== BM25 Index Status (점진적 업데이트 결과) ===')
for name, idx in [('Chunks', rag._bm25_chunks), ('Entities', rag._bm25_entities), ('Relations', rag._bm25_relations)]:
    built = idx and idx.is_built
    count = len(idx.corpus_ids) if built else 0
    print(f'  {name:12s}: {count} indexed')

## 4. 저장 데이터 확인

### 4a. 로컬 파일

In [ ]:
for f in sorted(os.listdir(WORK_DIR)):
    size = os.path.getsize(os.path.join(WORK_DIR, f))
    tag = ''
    if 'vdb_' in f: tag = '[VectorDB]'
    elif 'graph_' in f: tag = '[Graph]'
    elif 'kv_' in f: tag = '[KV]'
    print(f'  {f:50s} {size:>10,} bytes  {tag}')

### 4b. 지식 그래프 (LLM이 자동 추출)

In [ ]:
import networkx as nx

G = nx.read_graphml(os.path.join(WORK_DIR, 'graph_chunk_entity_relation.graphml'))
print(f'Nodes: {G.number_of_nodes()},  Edges: {G.number_of_edges()}')
print()
print('Entities (sample):')
for i, node in enumerate(sorted(G.nodes())):
    etype = G.nodes[node].get('entity_type', '?')
    print(f'  {node:35s}  [{etype}]')
    if i >= 19:
        print(f'  ... ({G.number_of_nodes() - 20} more)')
        break
print()
print('Relations (sample):')
for i, (src, tgt) in enumerate(G.edges()):
    print(f'  {src:30s} -> {tgt:30s}')
    if i >= 9:
        print(f'  ... ({G.number_of_edges() - 10} more)')
        break

### 4c. BM25 키워드 인덱스 (우리가 추가한 기능)

In [ ]:
bm25_c = rag._bm25_chunks
bm25_e = rag._bm25_entities
bm25_r = rag._bm25_relations

print(f'Chunks BM25:    {len(bm25_c.corpus_ids)} docs')
print(f'Entities BM25:  {len(bm25_e.corpus_ids)} docs')
print(f'Relations BM25: {len(bm25_r.corpus_ids)} docs')
print()

print('--- Entity index (sample) ---')
for eid in bm25_e.corpus_ids[:10]:
    print(f'  {eid}')
if len(bm25_e.corpus_ids) > 10:
    print(f'  ... ({len(bm25_e.corpus_ids) - 10} more)')

In [ ]:
# BM25 직접 쿼리 테스트
test_queries = ['chunking pipeline', 'API server', 'paragraph semantic', 'embedding']
print('=== BM25 Direct Query ===')
for q in test_queries:
    ent_hits = bm25_e.query(q, top_k=3)
    chunk_hits = bm25_c.query(q, top_k=2)
    print(f'\n  "{q}"')
    print(f'    Entities: {[r["id"] for r in ent_hits]}')
    print(f'    Chunks:   {[r["content"][:50]+"..." for r in chunk_hits]}')

## 5. 3가지 모드 쿼리 비교

| Mode | Method |
|---|---|
| `vector_only` | 벡터 유사도 검색만 |
| `keyword_only` | BM25 키워드 검색만 |
| `hybrid` | Vector + BM25 → RRF 병합 |

In [ ]:
async def compare_modes(query, mode='naive'):
    print('=' * 70)
    print(f'Query: "{query}"')
    print('=' * 70)
    
    param = QueryParam(mode=mode, top_k=5)
    results = {}
    
    for sm in ['vector_only', 'keyword_only', 'hybrid']:
        rag._addon_params['hybrid_search_mode'] = sm
        r = await rag.aquery_data(query, param=param)
        chunks = []
        if r.get('status') == 'success':
            chunks = r.get('data', {}).get('chunks', [])
        results[sm] = chunks
    
    labels = {'vector_only': 'Vector Only', 'keyword_only': 'Keyword (BM25)', 'hybrid': 'Hybrid (RRF)'}
    for sm in ['vector_only', 'keyword_only', 'hybrid']:
        chunks = results[sm]
        print(f'\n  [{labels[sm]}] {len(chunks)} chunks')
        for c in chunks[:3]:
            print(f'    - {c.get("content", "")[:80]}...')
        if not chunks:
            print('    (no results)')
    
    v = {c.get('content','')[:50] for c in results['vector_only']}
    k = {c.get('content','')[:50] for c in results['keyword_only']}
    only_k = k - v
    only_v = v - k
    if only_k: print(f'\n  -> BM25 found {len(only_k)} chunk(s) that Vector missed')
    if only_v: print(f'  -> Vector found {len(only_v)} chunk(s) that BM25 missed')
    print()
    rag._addon_params['hybrid_search_mode'] = 'hybrid'

In [ ]:
await compare_modes('file processing pipeline chunking strategy')

In [ ]:
await compare_modes('REST API server configuration')

In [ ]:
await compare_modes('paragraph semantic chunking')

In [ ]:
await compare_modes('How does LightRAG handle document ingestion and graph construction?')

## 6. RRF 시각화

In [ ]:
import pandas as pd
from lightrag.bm25_index import reciprocal_rank_fusion

vector_results = [
    {'id': 'DOC_A', 'score': 0.92},
    {'id': 'DOC_B', 'score': 0.88},
    {'id': 'DOC_C', 'score': 0.81},
    {'id': 'DOC_D', 'score': 0.75},
]
bm25_results = [
    {'id': 'DOC_C', 'score': 8.5},
    {'id': 'DOC_E', 'score': 6.2},
    {'id': 'DOC_A', 'score': 4.1},
    {'id': 'DOC_F', 'score': 3.8},
]

k = 60
vr = {r['id']: i for i, r in enumerate(vector_results, 1)}
br = {r['id']: i for i, r in enumerate(bm25_results, 1)}
rows = []
for doc in set(vr) | set(br):
    v, b = vr.get(doc), br.get(doc)
    vs = 1/(k+v) if v else 0
    bs = 1/(k+b) if b else 0
    rows.append({
        'Doc': doc,
        'Vec Rank': v or '-', 'BM25 Rank': b or '-',
        'Vec RRF': f'{vs:.5f}' if v else '-',
        'BM25 RRF': f'{bs:.5f}' if b else '-',
        'Total': f'{vs+bs:.5f}',
        'Source': 'BOTH' if (v and b) else ('Vec' if v else 'BM25'),
    })
df = pd.DataFrame(rows).sort_values('Total', ascending=False).reset_index(drop=True)
df.index += 1
df.index.name = 'Rank'
print('RRF(d) = sum(1/(k+rank)),  k=60')
print('BOTH sources → highest score')
print()
df

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fused = reciprocal_rank_fusion(vector_results, bm25_results, k=60)
docs = [r['id'] for r in fused]
vec_r = [vr.get(d, max(len(vector_results), len(bm25_results))+1) for d in docs]
bm25_r = [br.get(d, max(len(vector_results), len(bm25_results))+1) for d in docs]
fused_r = list(range(1, len(docs)+1))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(docs))
w = 0.25
ax.bar(x-w, vec_r, w, label='Vector Rank', color='#BBDEFB', edgecolor='#1565C0', lw=1.5)
ax.bar(x, bm25_r, w, label='BM25 Rank', color='#FFCCBC', edgecolor='#E64A19', lw=1.5)
ax.bar(x+w, fused_r, w, label='RRF Fused Rank', color='#C8E6C9', edgecolor='#2E7D32', lw=1.5)
ax.set_xlabel('Documents')
ax.set_ylabel('Rank (lower = better)')
ax.set_title('Vector vs BM25 vs Hybrid(RRF)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(docs, rotation=30, ha='right')
ax.legend()
ax.invert_yaxis()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
os.makedirs('images', exist_ok=True)
plt.savefig('images/ranking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cleanup

In [ ]:
await rag.finalize_storages()
print('Done.')

## Summary

| Mode | Method | Best for |
|---|---|---|
| `vector_only` | Cosine similarity | 의미 유사 문서, 서술형 쿼리 |
| `keyword_only` | BM25 TF-IDF | 고유명사, 정확한 키워드 매칭 |
| `hybrid` | Vector + BM25 + RRF | **Best of both** |

### 파이프라인 흐름
```
Insert:
  Raw Text → Chunking → [BM25 Chunks 인덱싱]
                       → Embedding (VDB 저장)
                       → LLM 엔티티/릴레이션 추출
                           → Entity VDB 저장 + [BM25 Entities 인덱싱]
                           → Relation VDB 저장 + [BM25 Relations 인덱싱]
                       → Graph 저장

Query:
  hybrid_search_mode → vector_only | keyword_only | hybrid(RRF)
```

### BM25 인덱싱 방식
- **점진적 업데이트**: 각 VDB upsert 시점마다 즉시 BM25 인덱스에 추가
- **초기화 시 full build**: `initialize_storages()` 에서 기존 데이터로 전체 빌드
- **_insert_done 보정**: 파이프라인 완료 시 누락 데이터 보정